In [1]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# ── 1. Pull data ──────────────────────────────────────────────────────────────
client = bigquery.Client(project="cbb6790-final-project")
query = """
SELECT *
FROM `cbb6790-final-project.analysis.CBB5790_FinalProject`
"""
df = client.query(query).to_dataframe()

# ── 2. Create binary target ───────────────────────────────────────────────────
df["long_stay"] = (df["icu_length_of_stay"] >= 7).astype(int)
df["gender_bin"] = (df["gender"] == "M").astype(int)

FEATURES = [
    "anchor_age",
    "creatinine_min", "creatinine_max",
    "bun_min", "bun_max",
    "potassium_min", "potassium_max",
    "bicarbonate_min", "bicarbonate_max",
    "sodium_min", "sodium_max",
    "mbp_min", "mbp_mean", "mbp_max",
    "heart_rate_min", "heart_rate_max",
    "urineoutput_24hr",
    "kdigo_stage",
    "gender_bin",
]
TARGET = "long_stay"

# ── 3. Stratified 80/10/10 split within each care unit ───────────────────────
train_dfs, tune_dfs, test_dfs = [], [], []
skipped_units = []

for unit in df["first_careunit"].dropna().unique():
    unit_df = df[df["first_careunit"] == unit].copy()
    if len(unit_df) < 30 or unit_df[TARGET].nunique() < 2:
        skipped_units.append((unit, len(unit_df), "too small or no outcome variation"))
        continue
    try:
        unit_trainval, unit_test = train_test_split(
            unit_df, test_size=0.10, random_state=42, stratify=unit_df[TARGET]
        )
        unit_train, unit_tune = train_test_split(
            unit_trainval, test_size=0.1111, random_state=42, stratify=unit_trainval[TARGET]
        )
        train_dfs.append(unit_train)
        tune_dfs.append(unit_tune)
        test_dfs.append(unit_test)
    except ValueError as e:
        skipped_units.append((unit, len(unit_df), str(e)))

df_train = pd.concat(train_dfs).reset_index(drop=True)
df_tune  = pd.concat(tune_dfs).reset_index(drop=True)
df_test  = pd.concat(test_dfs).reset_index(drop=True)

print("── Split sizes ──")
print(f"  Train: {len(df_train)} | Tune: {len(df_tune)} | Test: {len(df_test)}")

print("\n── Rows per care unit across splits ──")
for unit in df["first_careunit"].dropna().unique():
    n_train = (df_train["first_careunit"] == unit).sum()
    n_tune  = (df_tune["first_careunit"]  == unit).sum()
    n_test  = (df_test["first_careunit"]  == unit).sum()
    print(f"  {unit}: train={n_train}, tune={n_tune}, test={n_test}")

if skipped_units:
    print("\n── Skipped units ──")
    for unit, n, reason in skipped_units:
        print(f"  {unit} (n={n}): {reason}")

# ── 4. Helper functions ───────────────────────────────────────────────────────
def train_local_xgb(local_df, features, target, params):
    """Train a local XGBoost model and return raw booster + metadata."""
    imputer = SimpleImputer(strategy="median")
    X = imputer.fit_transform(local_df[features])
    y = local_df[target].values

    model = XGBClassifier(
        **params,
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=42,
    )
    model.fit(X, y)

    return {
        "booster":   model.get_booster(),  # raw booster for weight extraction
        "model":     model,
        "n_samples": len(local_df),
        "imputer":   imputer,
        "unit":      None,
    }


def federated_aggregate_xgb(client_results, features):
    """
    Average feature weights across all client boosters (weighted by n_samples).
    Returns a weight vector used for a soft ensemble prediction.
    """
    total_n = sum(r["n_samples"] for r in client_results)

    # Weighted average of feature importances as global signal
    agg_importance = np.zeros(len(features))
    for r in client_results:
        weight      = r["n_samples"] / total_n
        scores      = r["booster"].get_fscore()
        local_imp   = np.array([scores.get(f"f{i}", 0) for i in range(len(features))])
        # Normalize local importances
        if local_imp.sum() > 0:
            local_imp = local_imp / local_imp.sum()
        agg_importance += weight * local_imp

    return agg_importance


def predict_ensemble_xgb(client_results, eval_df, features):
    """
    Weighted ensemble prediction: each client model votes proportionally
    to its training set size.
    """
    total_n   = sum(r["n_samples"] for r in client_results)
    agg_probs = np.zeros(len(eval_df))

    for r in client_results:
        weight  = r["n_samples"] / total_n
        X       = r["imputer"].transform(eval_df[features])
        probs   = r["model"].predict_proba(X)[:, 1]
        agg_probs += weight * probs

    return agg_probs


def score_global_xgb(client_results, eval_df, features, target):
    probs = predict_ensemble_xgb(client_results, eval_df, features)
    return roc_auc_score(eval_df[target], probs)

# ── 5. Tune hyperparameters on validation set ─────────────────────────────────
param_grid = [
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1,  "subsample": 0.8},
    {"n_estimators": 100, "max_depth": 5, "learning_rate": 0.1,  "subsample": 0.8},
    {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.05, "subsample": 0.8},
    {"n_estimators": 200, "max_depth": 5, "learning_rate": 0.05, "subsample": 0.8},
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.01, "subsample": 1.0},
]
units = df_train["first_careunit"].dropna().unique()

print("\n── Tuning on validation set ──")
tune_results = {}

for params in param_grid:
    client_results = []
    for unit in units:
        unit_df = df_train[df_train["first_careunit"] == unit].copy()
        if len(unit_df) < 30 or unit_df[TARGET].nunique() < 2:
            continue
        result = train_local_xgb(unit_df, FEATURES, TARGET, params)
        result["unit"] = unit
        client_results.append(result)

    if not client_results:
        continue

    tune_auc = score_global_xgb(client_results, df_tune, FEATURES, TARGET)
    key = str(params)
    tune_results[key] = (tune_auc, params)
    print(f"  {params} → Tune AUC: {tune_auc:.4f}")

best_key    = max(tune_results, key=lambda k: tune_results[k][0])
best_params = tune_results[best_key][1]
print(f"\nBest params: {best_params} (Tune AUC: {tune_results[best_key][0]:.4f})")

# ── 6. Final federated training ───────────────────────────────────────────────
print(f"\n── Final federated training ──")
final_client_results = []

for unit in units:
    unit_df = df_train[df_train["first_careunit"] == unit].copy()
    if len(unit_df) < 30 or unit_df[TARGET].nunique() < 2:
        continue
    result = train_local_xgb(unit_df, FEATURES, TARGET, best_params)
    result["unit"] = unit
    final_client_results.append(result)
    print(f"  ✓ {unit}: n={result['n_samples']}")

global_importance = federated_aggregate_xgb(final_client_results, FEATURES)

# ── 7. AUC Evaluation ─────────────────────────────────────────────────────────
print("\n── Global Model AUC ──")
for split_name, split_df in [("Train", df_train), ("Tune", df_tune), ("Test", df_test)]:
    auc = score_global_xgb(final_client_results, split_df, FEATURES, TARGET)
    print(f"  {split_name}: {auc:.4f}")

print("\n── Per-Unit Local Model Test AUC ──")
for r in final_client_results:
    try:
        X     = r["imputer"].transform(df_test[FEATURES])
        probs = r["model"].predict_proba(X)[:, 1]
        auc   = roc_auc_score(df_test[TARGET], probs)
        print(f"  {r['unit']}: AUC = {auc:.4f}")
    except ValueError:
        print(f"  {r['unit']}: AUC could not be computed")

# ── 8. Feature importance ─────────────────────────────────────────────────────
print("\n── Global Feature Importance (weighted avg across clients) ──")
imp_df = pd.DataFrame({"feature": FEATURES, "importance": global_importance})
imp_df = imp_df.sort_values("importance", ascending=False)
print(imp_df.to_string(index=False))

── Split sizes ──
  Train: 24456 | Tune: 3061 | Test: 3061

── Rows per care unit across splits ──
  Medical Intensive Care Unit (MICU): train=7614, tune=952, test=952
  Surgical Intensive Care Unit (SICU): train=2509, tune=314, test=314
  Medical/Surgical Intensive Care Unit (MICU/SICU): train=5136, tune=642, test=642
  Trauma SICU (TSICU): train=1864, tune=234, test=234
  Coronary Care Unit (CCU): train=3661, tune=458, test=458
  Cardiac Vascular Intensive Care Unit (CVICU): train=2740, tune=343, test=343
  Neuro Surgical Intensive Care Unit (Neuro SICU): train=310, tune=39, test=39
  Neuro Intermediate: train=416, tune=52, test=52
  Intensive Care Unit (ICU): train=0, tune=0, test=0
  PACU: train=38, tune=5, test=5
  Neuro Stepdown: train=77, tune=10, test=10
  Surgery/Vascular/Intermediate: train=91, tune=12, test=12
  Medicine: train=0, tune=0, test=0
  Surgery/Trauma: train=0, tune=0, test=0
  Medicine/Cardiology Intermediate: train=0, tune=0, test=0

── Skipped units ──
  Intens

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:38:45] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:38:45] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:38:45] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:38:45] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:

  {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'subsample': 0.8} → Tune AUC: 0.8115


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:38:46] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:38:46] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:38:46] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:38:46] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:

  {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1, 'subsample': 0.8} → Tune AUC: 0.8107


...

  ✓ Coronary Care Unit (CCU): n=3661
  ✓ Cardiac Vascular Intensive Care Unit (CVICU): n=2740


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:38:56] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:38:56] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  ✓ Neuro Surgical Intensive Care Unit (Neuro SICU): n=310
  ✓ Neuro Intermediate: n=416
  ✓ PACU: n=38
  ✓ Neuro Stepdown: n=77


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:38:56] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:38:56] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:38:56] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:38:56] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  ✓ Surgery/Vascular/Intermediate: n=91

── Global Model AUC ──
  Train: 0.8363
  Tune: 0.8131
  Test: 0.8219

── Per-Unit Local Model Test AUC ──
  Medical Intensive Care Unit (MICU): AUC = 0.8141
  Surgical Intensive Care Unit (SICU): AUC = 0.8102
  Medical/Surgical Intensive Care Unit (MICU/SICU): AUC = 0.8048
  Trauma SICU (TSICU): AUC = 0.7830
  Coronary Care Unit (CCU): AUC = 0.8052
  Cardiac Vascular Intensive Care Unit (CVICU): AUC = 0.7974
  Neuro Surgical Intensive Care Unit (Neuro SICU): AUC = 0.7211
  Neuro Intermediate: AUC = 0.7457
  PACU: AUC = 0.6692
  Neuro Stepdown: AUC = 0.6168
  Surgery/Vascular/Intermediate: AUC = 0.6204

── Global Feature Importance (weighted avg across clients) ──
         feature  importance
     kdigo_stage    0.107587
urineoutput_24hr    0.099798
  heart_rate_max    0.074339
  creatinine_max    0.062017
        mbp_mean    0.060754
      anchor_age    0.058806
  heart_rate_min    0.055605
         mbp_min    0.053789
         bun_min    0.0509

In [2]:
# ── 9. Centralized Baseline (Pooled Data) ─────────────────────────────────────
print("\n── Centralized Baseline Training ──")

central_imputer = SimpleImputer(strategy="median")
X_train_central = central_imputer.fit_transform(df_train[FEATURES])
y_train_central = df_train[TARGET].values

central_model = XGBClassifier(
    **best_params,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42,
)
central_model.fit(X_train_central, y_train_central)

print("\n── Centralized Model AUC ──")
for split_name, split_df in [("Train", df_train), ("Tune", df_tune), ("Test", df_test)]:
    X_eval = central_imputer.transform(split_df[FEATURES])
    probs = central_model.predict_proba(X_eval)[:, 1]
    auc = roc_auc_score(split_df[TARGET], probs)
    print(f"  {split_name}: {auc:.4f}")

central_importances = central_model.feature_importances_
central_imp_df = pd.DataFrame({
    "feature": FEATURES,
    "importance": central_importances
}).sort_values("importance", ascending=False)

print("\n── Centralized Feature Importance ──")
print(central_imp_df.to_string(index=False))


── Centralized Baseline Training ──


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:39:00] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



── Centralized Model AUC ──
  Train: 0.8396
  Tune: 0.8139
  Test: 0.8208

── Centralized Feature Importance ──
         feature  importance
     kdigo_stage    0.475335
urineoutput_24hr    0.051248
  creatinine_max    0.050372
      anchor_age    0.043587
  creatinine_min    0.042621
  heart_rate_max    0.035031
      sodium_max    0.033182
      gender_bin    0.031024
        mbp_mean    0.030631
         mbp_min    0.026310
 bicarbonate_min    0.025852
 bicarbonate_max    0.024624
   potassium_min    0.024208
         mbp_max    0.020591
         bun_min    0.018363
   potassium_max    0.017782
      sodium_min    0.017739
  heart_rate_min    0.016282
         bun_max    0.015217
